### Imports

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

sys.path.append("..")

from src.data.load_data import load_tables
from src.data.joins import build_dataset, join_summary
from src.features.temporal_features import add_temporal_features
from src.features.cyclic_features import add_cyclic_features
from src.features.historical_features import add_historical_features
from src.features.encoding import FrequencyEncoder, TargetEncoder

SEED       = 42
OUTPUT_DIR = Path().resolve().parent / 'data' / 'processed'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style='whitegrid')
print('Imports OK')

### Load tables

In [ ]:
accidents, weather, raw = load_tables()

### Binary Target Construction

In [ ]:
df = build_dataset(accidents, weather)
join_summary(df)

In [ ]:
count = df['target'].value_counts()
fig, ax = plt.subplots(figsize=(5, 3))
count.plot(kind='bar', ax=ax, color=['steelblue','salmon'], edgecolor='white')
ax.set_xticklabels(['Without Accident (0)', 'With Accident (1)'], rotation=0)
ax.set_title('Distribution of Target Variable')
for i, v in enumerate(count):
    ax.text(i, v + 50, f'{v:,}\n({v/len(df)*100:.1f}%)', ha='center', fontsize=9)
plt.tight_layout()
plt.show()
print(f'\n Positive class: {df["target"].mean()*100:.2f}% ? imbalanced problem.')

### Temporal Variables

In [ ]:
df = add_temporal_features(df)
print('Nuevas columnas:', ['hour','weekday_num','is_weekend','is_holiday','time_period'])
df[['TW','hour','weekday_num','is_weekend','is_holiday','time_period']].head(4)

### Cyclical variables

In [ ]:
df = add_cyclic_features(df)
print('New features added: hour_sin/cos, weekday_sin/cos, dayOfYear_sin/cos, month_sin/cos')

hours = pd.DataFrame({'hour': range(24)})
hours['sin'] = np.sin(2*np.pi*hours['hour']/24)
hours['cos'] = np.cos(2*np.pi*hours['hour']/24)
fig, ax = plt.subplots(figsize=(6,4))
ax.scatter(hours['cos'], hours['sin'], c=hours['hour'], cmap='hsv', s=60)
for _, r in hours.iterrows():
    ax.annotate(int(r['hour']), (r['cos'], r['sin']), fontsize=7)
ax.set_title('Hour of Day as Cyclic Features')
ax.set_xlabel('cos'); ax.set_ylabel('sin')
plt.tight_layout(); plt.show()

### Historycal Variables

In [ ]:
df = add_historical_features(df, raw)
df[['TW','BARRIO','target',
    'hist_acc_neighborhood_total','hist_acc_neighborhood_30d',
    'hist_acc_hour_neighborhood','hist_rate_neighborhood']].head(6)

### Define Data Columns

In [ ]:
DROP_COLS = ['TW', 'target']

COLS_TARGET_ENC = ['BARRIO']
COLS_FREQ_ENC   = ['summary', 'icon', 'time_period']

NUMERIC_COLS = [
    'Lat', 'Lon',
    'hour', 'weekday_num', 'is_weekend', 'is_holiday',
    'hour_sin', 'hour_cos',
    'weekday_sin', 'weekday_cos',
    'dayOfYear_sin', 'dayOfYear_cos',
    'month_sin', 'month_cos',
    'temperature', 'apparentTemperature', 'dewPoint', 'humidity',
    'precipIntensity', 'precipProbability',
    'windSpeed', 'windBearing', 'cloudCover', 'uvIndex', 'visibility',
    'hist_acc_neighborhood_total', 'hist_acc_neighborhood_30d',
    'hist_acc_hour_neighborhood', 'hist_rate_neighborhood',
]

NUMERIC_COLS    = [c for c in NUMERIC_COLS    if c in df.columns]
COLS_TARGET_ENC = [c for c in COLS_TARGET_ENC if c in df.columns]
COLS_FREQ_ENC   = [c for c in COLS_FREQ_ENC   if c in df.columns]

print(f'Num?ricas:      {len(NUMERIC_COLS)}')
print(f'Columnas:       {NUMERIC_COLS}')
print(f'Target Enc:     {COLS_TARGET_ENC}')
print(f'Frequency Enc:  {COLS_FREQ_ENC}')

### Split temporal train / test

El split se realiza ordenando el dataset por `TW` y reservando el 20 % final como test.
Esto garantiza que el conjunto de prueba contenga s?lo datos posteriores a los usados en entrenamiento, evitando fuga temporal.
La selecci?n de umbral y la validaci?n interna se realizan en `05_validation_model_selection.ipynb`; aqu? s?lo se prepara el train/test temporal.


In [ ]:
# Orden temporal por TW para garantizar un split train/test honesto
# El 80 % inicial es train y el 20 % final es test (sin shuffle).
df_sorted = df.sort_values('TW').reset_index(drop=True)

cut = int(len(df_sorted) * 0.8)
train = df_sorted.iloc[:cut].copy()
test  = df_sorted.iloc[cut:].copy()

feature_cols = NUMERIC_COLS + COLS_TARGET_ENC + COLS_FREQ_ENC
feature_cols = [c for c in feature_cols if c in df.columns]

X_train = train[feature_cols]
y_train = train['target']
X_test  = test[feature_cols]
y_test  = test['target']

print(f'Train: {X_train.shape}  ({train["TW"].min().date()} ? {train["TW"].max().date()})')
print(f'Test : {X_test.shape}   ({test["TW"].min().date()} ? {test["TW"].max().date()})')
print(f'\nTarget train: {y_train.mean()*100:.2f}% positives')
print(f'Target test : {y_test.mean()*100:.2f}% positives')

### Encoding - Fit in train, Transform in test

In [ ]:
# Ajustar encoders s?lo con datos de entrenamiento para evitar fuga desde el test.
# Transformar el test usando los objetos ya fit en train.
te = TargetEncoder(columns=COLS_TARGET_ENC, smoothing=10)
X_train = te.fit_transform(X_train, y_train)
X_test  = te.transform(X_test)

fe = FrequencyEncoder(columns=COLS_FREQ_ENC)
X_train = fe.fit_transform(X_train)
X_test  = fe.transform(X_test)

X_train = X_train.drop(columns=COLS_TARGET_ENC + COLS_FREQ_ENC, errors='ignore')
X_test  = X_test.drop(columns=COLS_TARGET_ENC + COLS_FREQ_ENC, errors='ignore')

print(f'X_train tras encoding: {X_train.shape}')
print(f'X_test  tras encoding: {X_test.shape}')

### Scaling and Imputation Pipeline

In [ ]:
numeric_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
])

X_train_proc = numeric_pipe.fit_transform(X_train)
X_test_proc  = numeric_pipe.transform(X_test)

print(f'X_train_proc: {X_train_proc.shape}')
print(f'X_test_proc : {X_test_proc.shape}')
print(f'Remaining Nulls: {np.isnan(X_train_proc).sum()}')

### Checking and Saving

In [ ]:
feature_names = X_train.columns.tolist()
print(feature_names)
df_check = pd.DataFrame(X_train_proc, columns=feature_names)

print('Statistics after scaling (should be ~0 mean, ~1 std):')
display(df_check[['hour_sin', 'hour_cos', 'temperature', 'hist_rate_neighborhood']]
        .describe().round(3))

In [ ]:
np.save(OUTPUT_DIR / 'X_train.npy', X_train_proc)
np.save(OUTPUT_DIR / 'X_test.npy',  X_test_proc)
y_train.reset_index(drop=True).to_csv(OUTPUT_DIR / 'y_train.csv', index=False)
y_test.reset_index(drop=True).to_csv( OUTPUT_DIR / 'y_test.csv',  index=False)

joblib.dump(numeric_pipe, OUTPUT_DIR / 'numeric_pipeline.joblib')
joblib.dump(te,           OUTPUT_DIR / 'target_encoder.joblib')
joblib.dump(fe,           OUTPUT_DIR / 'freq_encoder.joblib')

pd.Series(feature_names).to_csv(OUTPUT_DIR / 'feature_names.csv', index=False)

print('Artifacts saved in:', OUTPUT_DIR)
print([f.name for f in OUTPUT_DIR.iterdir()])